In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 21. RLHF — テキスト テキスト テキスト学習 ⭐ [CPU/GPU ベンチマーク ⑨]

> **学習目標**
> - テキスト学習 テキスト テキスト(MDP, テキスト, テキスト, テキスト)テキスト LLMテキスト テキスト
> - テキスト テキスト(Policy Gradient)テキスト REINFORCE テキスト テキスト度テキスト
> - PPOテキスト clipped objectiveテキスト テキスト テキスト
> - テキスト モデル(RM) 学習テキスト テキスト

## 21.1 SFTテキスト テキスト テキスト 問題

SFT モデルテキスト "テキスト テキスト"テキスト テキスト:
- テキスト **テキスト** テキスト テキスト
- テキスト テキスト度 テキスト テキスト テキスト テキスト
- "テキスト(alignment)" テキスト: モデルテキスト テキスト テキスト テキスト

## 21.2 RLHF 3テキスト

1. **SFT**: テキスト テキスト (Ch 20)
2. **Reward Model (RM)**: テキスト テキスト テキスト テキスト モデル 学習
3. **PPO**: RMテキスト テキスト テキスト(=LLM) テキスト学習

## 21.3 テキスト学習 テキスト — MDP

**MDP** (Markov Decision Process): $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$
- $\mathcal{S}$: テキスト 空間
- $\mathcal{A}$: テキスト 空間
- $P(s'|s, a)$: テキスト テキスト
- $R(s, a)$: テキスト
- $\gamma$: テキスト

LLM テキスト:
- テキスト $s_t$ = テキスト テキスト テキスト $[w_0, \ldots, w_{t-1}]$
- テキスト $a_t$ = テキスト テキスト $w_t$
- テキスト $\pi_\theta(a_t | s_t) = P(w_t | w_{<t}; \theta)$
- テキスト $R$ = RMテキスト テキスト テキスト テキスト (テキスト テキスト テキスト)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# テキスト テキスト (toy)
vocab_size = 100  # テキスト テキスト テキスト テキスト
print(f"vocab_size (toy): {vocab_size}")

## 21.4 テキスト テキスト テキスト (REINFORCE)

テキスト: $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)]$ テキスト.

$$\nabla J(\theta) = \mathbb{E}_{\tau} \left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t\right]$$

- $G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$: テキスト テキスト テキスト
- $\nabla \log \pi$: テキスト テキスト テキスト → "テキスト テキスト テキスト テキスト テキスト" テキスト
- $G_t$テキスト テキスト → テキスト テキスト テキスト テキスト テキスト テキスト

**Advantage** $A_t = G_t - V(s_t)$: テキスト $V$ テキスト テキスト テキスト.


In [ ]:
# REINFORCE テキスト (テキスト toy MDP)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# テキスト: テキスト bandit (3テキスト テキスト, テキスト 分布 テキスト)
true_rewards = [0.2, 0.5, 0.8]  # テキスト 2テキスト テキスト

class PolicyNet(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(1, 16)
        self.head = nn.Linear(16, n_actions)
    def forward(self, x):
        return self.head(F.relu(self.fc(x)))

policy = PolicyNet(3)
opt = torch.optim.Adam(policy.parameters(), lr=0.01)

n_episodes = 500
all_rewards = []
for ep in range(n_episodes):
    # テキスト: 1 step (bandit)
    state = torch.tensor([[1.0]])
    logits = policy(state)
    probs = F.softmax(logits, dim=-1)
    action = torch.multinomial(probs, 1).item()
    reward = true_rewards[action] + np.random.randn() * 0.05

    # REINFORCE: log_prob * reward
    log_prob = F.log_softmax(logits, dim=-1)[0, action]
    loss = -log_prob * reward  # テキスト → テキスト テキスト
    opt.zero_grad()
    loss.backward()
    opt.step()
    all_rewards.append(reward)

print(f"テキスト 50 テキスト Mean Reward: {np.mean(all_rewards[:50]):.3f}")
print(f"テキスト 50 テキスト Mean Reward: {np.mean(all_rewards[-50:]):.3f}")
print(f"Theory Maximum: {max(true_rewards):.3f}")


## 21.5 PPO — Trust Regionテキスト Clipped Objective

REINFORCEテキスト テキスト テキスト. PPOテキスト テキスト テキスト テキスト テキスト テキスト テキスト テキスト.

**テキスト**: $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$

**Clipped Objective**:
$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[\min\left(r_t \hat{A}_t, \, \mathrm{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat{A}_t\right)\right]$$

- $\epsilon \approx 0.2$: テキスト テキスト テキスト
- minテキスト clip → テキスト テキスト テキスト テキスト テキスト テキスト
- trust regionテキスト テキスト


In [ ]:
# PPO テキスト (toy 問題)
import torch
import torch.nn as nn
import torch.nn.functional as F

class Policy(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.fc = nn.Linear(4, 32)
        self.head = nn.Linear(32, n_actions)
    def forward(self, x):
        return self.head(F.tanh(self.fc(x)))

# テキスト テキスト (テキスト)
def env_step(state, action):
    reward = float((state.sum() + action - 2) / 10)  # テキスト テキスト
    next_state = state + torch.randn(4) * 0.1
    return next_state, reward, False

policy = Policy(3)
opt = torch.optim.Adam(policy.parameters(), lr=3e-4)

# PPO テキスト
def ppo_update(policy, opt, states, actions, old_log_probs, advantages, returns, values,
               epsilon=0.2, n_epochs=4):
    for _ in range(n_epochs):
        logits = policy(states)
        new_log_probs = F.log_softmax(logits, dim=-1)
        new_log_probs_a = new_log_probs.gather(1, actions.unsqueeze(1)).squeeze()

        ratio = torch.exp(new_log_probs_a - old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()

        # テキスト 関数 テキスト (テキスト)
        critic_loss = F.mse_loss(values.squeeze(), returns)

        loss = actor_loss + 0.5 * critic_loss
        opt.zero_grad()
        loss.backward()
        opt.step()

# 学習 テキスト
n_iterations = 50
all_rewards = []
for it in range(n_iterations):
    # rollout テキスト
    states, actions, rewards, old_log_probs = [], [], [], []
    state = torch.randn(1, 4)
    ep_reward = 0
    for step in range(20):
        with torch.no_grad():
            logits = policy(state)
            probs = F.softmax(logits, dim=-1)
            dist = torch.distributions.Categorical(probs)
            action = dist.sample()
            old_log_prob = dist.log_prob(action)
        next_state, reward, done = env_step(state.squeeze(), action.item())
        states.append(state.squeeze())
        actions.append(action)
        rewards.append(reward)
        old_log_probs.append(old_log_prob)
        ep_reward += reward
        state = next_state.unsqueeze(0)
    all_rewards.append(ep_reward / 20)

    # advantage 計算 (テキスト: テキスト テキスト)
    advantages = torch.tensor(rewards)
    returns = torch.tensor(rewards)
    values = torch.zeros_like(returns)  # テキスト テキスト (テキスト)

    ppo_update(policy, opt, torch.stack(states), torch.tensor(actions),
               torch.stack(old_log_probs), advantages, returns, values)

print(f"テキスト 10 テキスト Mean Reward: {np.mean(all_rewards[:10]):.4f}")
print(f"テキスト 10 テキスト Mean Reward: {np.mean(all_rewards[-10:]):.4f}")


## 21.6 テキスト モデル (RM) 学習

RMテキスト テキスト テキスト テキスト モデル. **テキスト データ** (chosen vs rejected)テキスト 学習.

**Bradley-Terry モデル**:
$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

テキスト (NLL):
$$\mathcal{L}_{\text{RM}} = -\log \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

- $\mathbf{y}_w$: テキスト(chosen) テキスト
- $\mathbf{y}_l$: テキスト(rejected) テキスト
- $r(\mathbf{x}, \mathbf{y})$: RMテキスト テキスト テキスト

RM テキスト: SFT モデルテキスト テキスト 出力テキスト テキスト.


In [ ]:
# RM 学習 テキスト
import torch
import torch.nn as nn
import torch.nn.functional as F

class RewardModel(nn.Module):
    def __init__(self, vocab_size, d_model=32):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.rnn = nn.LSTM(d_model, d_model, batch_first=True)
        self.head = nn.Linear(d_model, 1)  # テキスト テキスト

    def forward(self, x):
        emb = self.emb(x)
        _, (h, _) = self.rnn(emb)
        return self.head(h.squeeze(0))  # (B,)

# toy テキスト データ
n_samples = 100
torch.manual_seed(0)
rm = RewardModel(vocab_size=vocab_size, d_model=32)
opt = torch.optim.AdamW(rm.parameters(), lr=1e-3)

# テキスト chosen/rejected pairs (テキスト)
print("RM Training テキスト:")
losses = []
for step in range(200):
    # テキスト データ: テキスト テキスト
    chosen = torch.randint(0, vocab_size, (8, 16))
    rejected = torch.randint(0, vocab_size, (8, 16))
    # テキスト テキスト: chosenテキスト テキスト IDテキスト テキスト テキスト テキスト
    # (テキスト RMテキスト 学習テキスト テキスト)
    r_chosen = rm(chosen).squeeze(-1)
    r_rejected = rm(rejected).squeeze(-1)
    loss = -F.logsigmoid(r_chosen - r_rejected).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('RM Loss')
plt.title('Reward Model Training (Bradley-Terry)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch21_rm_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"テキスト RM loss: {losses[-1]:.4f} (Theory Minimum ≈ 0.69 = -log(0.5) テキスト)")


## 21.7 KL テキスト — テキスト モデルテキスト テキスト テキスト テキスト度テキスト

RLHFテキスト RM テキスト テキスト モデルテキスト テキスト テキスト テキスト テキスト テキスト (reward hacking).

テキスト: テキスト モデル(SFT モデル)テキスト テキスト テキスト テキスト KL テキスト.

$$R_{\text{total}} = R_{\text{RM}}(\mathbf{x}, \mathbf{y}) - \beta \, D_{\text{KL}}(\pi_\theta(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

- $\beta$: KL テキスト度 (テキスト 0.01~0.5)
- $\pi_{\text{ref}}$: SFT モデル (テキスト)
- テキスト KL: $\sum_t \log \frac{\pi_\theta(w_t|...)}{\pi_{\text{ref}}(w_t|...)}$


In [ ]:
# KL テキスト 計算 (テキスト vs テキスト)
def token_kl_div(policy_logits, ref_logits):
    """テキスト KL Divergence."""
    p = F.softmax(policy_logits, dim=-1)
    log_p = F.log_softmax(policy_logits, dim=-1)
    log_q = F.log_softmax(ref_logits, dim=-1)
    return (p * (log_p - log_q)).sum(dim=-1)  # (B, T)

# テキスト
torch.manual_seed(0)
B, T, V = 4, 16, 100
ref_logits = torch.randn(B, T, V)
policy_logits = ref_logits + torch.randn(B, T, V) * 0.5  # テキスト テキスト Policy

kl = token_kl_div(policy_logits, ref_logits)
print(f"KL テキスト shape: {kl.shape}")
print(f"Mean KL: {kl.mean():.4f}")
print(f"Maximum KL: {kl.max():.4f}")
print("\n=> Policyテキスト テキスト テキスト KL Increase. RLHFテキスト テキスト テキスト テキスト.")


## 21.8 [CPU/GPU ベンチマーク ⑨] SFT vs RLHF 学習 比較

RLHFテキスト モデル 4テキスト テキスト: テキスト, テキスト, RM, テキスト関数. テキスト·時間 3~4テキスト.


In [ ]:
# RLHF テキスト テキスト (テキスト)
print("RLHF Trainingテキスト テキスト Model:")
print("  1. Policy model (Trainingテキスト)")
print("  2. Reference model (テキスト, KLテキスト)")
print("  3. Reward model (テキスト)")
print("  4. Value/Critic model (Trainingテキスト, optional)")
print()

# テキスト: 7B LLaMA RLHF
model_size_gb = 7 * 2 * 4  # 7B params * 2 bytes (FP16) * 4 models? No:
# テキスト: policy (FP32 grad, FP32 opt state)
# AdamW: param + grad + m + v = 4テキスト
# 7B モデル FP16: 14GB
# AdamW テキスト FP32: 7B * 8 bytes = 56GB
# policy: 14 + 56 = 70GB
# reference: 14GB
# RM: 14GB
# value: 14 + 56 = 70GB
# テキスト: テキスト 168GB → A100 80GB 2テキスト テキスト

print("7B Model RLHF Memory Estimation:")
print(f"  Policy (FP16 + AdamW FP32): ~70GB")
print(f"  Reference (FP16, テキスト): ~14GB")
print(f"  Reward Model (FP16, テキスト): ~14GB")
print(f"  Value (FP16 + AdamW): ~70GB")
print(f"  テキスト: ~168GB (A100 80GB × 2 テキスト)")
print("\n=> RLHFテキスト テキスト テキスト. DPO(Ch 22)テキスト テキスト Efficiencyテキスト テキスト.")


## 21.9 RLHFテキスト テキスト

- **テキスト テキスト(reward hacking)**: RM テキスト テキスト テキスト テキスト テキスト
- **テキスト テキスト(mode collapse)**: テキスト テキスト テキスト テキスト テキスト
- **テキスト**: PPO テキスト テキスト

## 21.10 要点

| テキスト | テキスト | テキスト |
|---|---|---|
| テキスト テキスト | $\nabla J = \mathbb{E}[\nabla\log\pi \cdot G]$ | REINFORCE |
| Advantage | $A = G - V$ | テキスト テキスト |
| PPO clip | $\min(rA, \text{clip}(r)A)$ | trust region |
| RM テキスト | $-\log\sigma(r_w - r_l)$ | Bradley-Terry |
| KL テキスト | $R - \beta D_{KL}$ | テキスト テキスト |

## 演習問題

1. REINFORCEテキスト CartPole テキスト toy テキスト 学習テキスト.
2. PPOテキスト $\epsilon = 0.1, 0.2, 0.3$テキスト 比較テキスト テキスト テキスト.
3. RM 学習テキスト chosenテキスト rejectedテキスト テキスト テキスト vs テキスト テキスト lossテキスト 比較テキスト.
4. KL テキスト $\beta = 0, 0.1, 1.0$テキスト テキスト テキスト テキスト.
5. RLHF テキスト DPOテキスト テキスト テキスト テキスト テキスト.

> 解答: `solutions/ch21_solutions.ipynb`
